In [1]:
import os

session_path = "C:/Users/Acer/0. CAPSTONE/dataset/301_P"

print(" Danh sách file trong session 301_P:")
for file in os.listdir(session_path):
    print("-", file)


 Danh sách file trong session 301_P:
- .ipynb_checkpoints
- 301_AUDIO.wav
- 301_CLNF_AUs.txt
- 301_CLNF_features.txt
- 301_CLNF_features3D.txt
- 301_CLNF_gaze.txt
- 301_CLNF_hog.bin
- 301_CLNF_pose.txt
- 301_COVAREP.csv
- 301_FORMANT.csv
- 301_TRANSCRIPT.csv


In [2]:
import numpy as np
import pandas as pd
from tqdm import tqdm

#  Khai báo thư mục dataset đầu vào và nơi lưu đầu ra
video_dataset_dir = "C:/Users/Acer/0. CAPSTONE/dataset"
video_output_path = "C:/Users/Acer/0. CAPSTONE/embeddings/video_embeddings"
video_embedding_dict = {}

#  Các file đặc trưng video cần xử lý
video_feature_types = {
    "CLNF_features": ".txt",
    "CLNF_features3D": ".txt",
    "CLNF_gaze": ".txt",
    "CLNF_pose": ".txt",
    "CLNF_AUs": ".txt",
    "CLNF_hog": ".bin"
}

def extract_stats_from_txt(file_path):
    try:
        df = pd.read_csv(file_path, sep=r'\s+|,', engine='python', comment='#')
        if df.empty:
            return None
        return np.concatenate([
            df.mean(skipna=True).values,
            df.std(skipna=True).values,
            df.min(skipna=True).values,
            df.max(skipna=True).values,
        ])
    except Exception as e:
        print(f"[ TXT lỗi] {file_path}: {e}")
        return None

def extract_stats_from_hog(file_path):
    try:
        with open(file_path, 'rb') as f:
            header = np.fromfile(f, dtype=np.int32, count=2)  # [n_frames, n_dim]
            n_frames, n_dim = header
            hog_data = np.fromfile(f, dtype=np.float32, count=n_frames * n_dim).reshape((n_frames, n_dim))
        return np.concatenate([
            np.mean(hog_data, axis=0),
            np.std(hog_data, axis=0),
            np.min(hog_data, axis=0),
            np.max(hog_data, axis=0),
        ])
    except Exception as e:
        print(f"[ HOG lỗi] {file_path}: {e}")
        return None

#  Duyệt toàn bộ session
all_sessions = sorted([s for s in os.listdir(video_dataset_dir) if os.path.isdir(os.path.join(video_dataset_dir, s)) and s.endswith("_P")])

print(f" Tổng số session phát hiện: {len(all_sessions)}")

for session in tqdm(all_sessions, desc=" Đang xử lý video sessions"):
    session_path = os.path.join(video_dataset_dir, session)
    session_id = session.split("_")[0]

    all_features = []
    missing_file = False

    for feature_key, ext in video_feature_types.items():
        filename = f"{session_id}_{feature_key}{ext}"
        file_path = os.path.join(session_path, filename)

        if not os.path.exists(file_path):
            print(f"[ Thiếu file] {file_path}")
            missing_file = True
            break

        if feature_key == "CLNF_hog":
            stats = extract_stats_from_hog(file_path)
        else:
            stats = extract_stats_from_txt(file_path)

        if stats is not None:
            all_features.append(stats)
        else:
            print(f"[ Không thể trích đặc trưng] {file_path}")
            missing_file = True
            break

    if not missing_file:
        video_embedding = np.concatenate(all_features)
        video_embedding_dict[session_id] = video_embedding

#  Lưu
if video_embedding_dict:
    os.makedirs(os.path.dirname(video_output_path), exist_ok=True)
    np.save(video_output_path, video_embedding_dict)
    print(f" Đã lưu video embeddings vào: {video_output_path}")
    print(f" Tổng số session đã lưu: {len(video_embedding_dict)}")
    print(" Ví dụ các session đầu:")
    for i, (sid, emb) in enumerate(video_embedding_dict.items()):
        print(f"  {i+1}. {sid}: shape = {emb.shape}")
        if i == 6: break
else:
    print(" Không có session nào được lưu.")


 Tổng số session phát hiện: 11


 Đang xử lý video sessions: 100%|██████████| 11/11 [03:30<00:00, 19.16s/it]

 Đã lưu video embeddings vào: C:/Users/Acer/0. CAPSTONE/embeddings/video_embeddings
 Tổng số session đã lưu: 11
 Ví dụ các session đầu:
  1. 301: shape = (3212,)
  2. 302: shape = (3212,)
  3. 303: shape = (3212,)
  4. 304: shape = (3212,)
  5. 305: shape = (3212,)
  6. 306: shape = (3212,)
  7. 308: shape = (3212,)


In [3]:
import shutil

src_path = "C:/Users/Acer/0. CAPSTONE/embeddings/video_embeddings.npy"
dst_path = "C:/Users/Acer/0. CAPSTONE/embeddings/video_embeddings/video_embeddings.npy"

shutil.move(src_path, dst_path)
print(" Đã chuyển file vào thư mục video_embeddings/")


 Đã chuyển file vào thư mục video_embeddings/
